# Chess Machine Learning Application

This notebook implements a chess application that replaces the traditional Minimax algorithm with a machine learning model for position evaluation.

### Architecture & Usage
- **ML Model**: Uses a Multi-Layer Perceptron (MLP) trained on millions of chess positions (from Kaggle dataset).
- **Input Features**: Converts board state (FEN) into a 781-dimensional vector (piece placement, castling rights, etc.).
- **Evaluation**: The model predicts a score for any given board state, representing the advantage for White.
- **Move Selection**: The engine generates all legal moves, evaluates the resulting positions using the ML model, and selects the move with the highest favorable score for the current player.

To use this application:
1. Run the installation cell to set up dependencies.
2. Run the main code cell to launch the interactive chess GUI.
3. Play against the AI or analyze positions using the "AI Analysis" panel.

In [ ]:
!pip -q install chess ipywidgets

### Code Overview & Key Features

The code below sets up the interactive chess environment and integrates the ML engine:

- **Libraries**: 
  - `chess`: Python-chess library for move generation and board state management.
  - `ipywidgets`: For creating the interactive UI (buttons, board display).
  - `ml_chess_engine`: Custom module that handles loading the model and running evaluations.

- **Key Components**:
  - `evaluate_board_raw(board)`: Interface between the game and the ML model.
  - `best_move(board)`: Decides the best move by scoring all candidates via the ML model.
  - `ChessApp`: The main class handling the game loop, UI events, and board rendering.
  - `render_analysis_panel()`: visualizes the top candidate moves and their ML-predicted scores.

In [ ]:
import math
from dataclasses import dataclass
from typing import Optional, Tuple, List, Dict

import chess
import ipywidgets as W
from IPython.display import display, HTML

# ML engine
from ml_chess_engine import load_model, evaluate_board_ml, suggest_move_ml
MODEL = load_model("chess_eval_mlp.joblib")

# -------------------------------
#           UI Styles
# -------------------------------
ANALYSIS_CSS = """
<style>
.ai-panel{border:1px solid #c8b8a8;border-radius:8px;background:#f9f6f0;padding:8px;margin-top:8px;max-width:360px}
.ai-panel h4{margin:4px 0 8px 0;font-weight:600}
.ai-best{background:#e7f6e7;border-radius:6px;padding:6px 8px;margin-bottom:8px}
.ai-list .row{display:flex;justify-content:space-between;padding:4px 8px;border-radius:4px}
.ai-list .row:nth-child(odd){background:#f2f7f2}
.ai-move{font-weight:600}
.ai-score{opacity:.85}
</style>
"""
display(HTML("<style>.widget-button{font-size:22px !important;}</style>"+ANALYSIS_CSS))

# -------------------------------
#           ENGINE (ML-only)
# -------------------------------

def evaluate_board_raw(board: chess.Board) -> int:
    """Return centipawns from the PERSPECTIVE OF THE SIDE TO MOVE."""
    return int(round(evaluate_board_ml(board, MODEL, return_centipawns=True)))

def best_move(board: chess.Board, ai_color: chess.Color) -> Optional[chess.Move]:
    """1-ply ML: evaluate child positions and choose the best for the side to move."""
    mv, _ = suggest_move_ml(board, MODEL)
    return mv

# -------------------------------
#     AI Analysis (ML scoring)
# -------------------------------

def _score_all_moves_ml(board: chess.Board, top_n: int = 20):
    """Return a sorted list [(move, score_cp)] for side to move, using 1-ply ML.
       Handles terminal nodes so mates are always preferred/avoided."""
    rows = []
    player = board.turn
    for mv in board.legal_moves:
        board.push(mv)
        if board.is_game_over():
            outcome = board.outcome()
            if outcome is None or outcome.winner is None:
                sc = 0
            else:
                sc = 10_000_000 if outcome.winner == player else -10_000_000
        else:
            cp = evaluate_board_ml(board, MODEL, return_centipawns=True)   # side-to-move in child
            sc = int(round(cp if player == chess.WHITE else -cp))          # current player's perspective
        board.pop()
        rows.append((mv, sc))
    rows.sort(key=lambda x: x[1], reverse=True)
    return rows[:top_n]

def render_analysis_panel(board: chess.Board, top_n: int = 20) -> str:
    rows = _score_all_moves_ml(board, top_n=top_n)
    if not rows:
        return "<div class='ai-panel'><h4>AI Analysis</h4><div>No legal moves.</div></div>"
    best_mv, best_sc = rows[0]
    best_san = board.san(best_mv)
    items = []
    for i,(mv,sc) in enumerate(rows,1):
        items.append(
            f"<div class='row'><div>{i}. <span class='ai-move'>{board.san(mv)}</span></div>"
            f"<div class='ai-score'>{sc:+d}</div></div>"
        )
    return f"""
    <div class='ai-panel'>
      <h4>AI Analysis</h4>
      <div class='ai-best'><b>Best Move:</b> {best_san} <span class='ai-score'>(Score: {best_sc:+d})</span></div>
      <div class='ai-list'>
        <div style='margin:4px 8px 6px 8px;opacity:.9'>All Moves ({len(rows)} total):</div>
        {''.join(items)}
      </div>
    </div>
    """

# -------------------------------
#           UI (ipywidgets)
# -------------------------------

# Piece glyphs
GLYPH = {'P':'♙','N':'♘','B':'♗','R':'♖','Q':'♕','K':'♔',
         'p':'♟','n':'♞','b':'♝','r':'♜','q':'♛','k':'♚'}
LIGHT = '#F0D9B5'; DARK  = '#B58863'; SEL   = '#f6f67a'; TARGET= '#b9e6a1'; CAPT  = '#f5a3a3'

@dataclass
class GameState:
    board: chess.Board
    ai_color: Optional[chess.Color]  # None for 2-player mode
    orientation_white: bool          # True = white-bottom view

    def status_text(self) -> str:
        if self.board.is_game_over():
            outcome = self.board.outcome()
            if outcome is None or outcome.winner is None:
                return "Game over · Draw"
            return "Game over · " + ("White wins" if outcome.winner == chess.WHITE else "Black wins")
        who = "White" if self.board.turn == chess.WHITE else "Black"
        return f"Turn: {who}" + (" · Check!" if self.board.is_check() else "")

class ChessApp:
    def __init__(self):
        # Controls (no depth slider)
        self.mode = W.ToggleButtons(options=[('Vs AI','ai'),('Two Players','2p')], value='ai', description='Mode:')
        self.color_sel = W.ToggleButtons(options=[('White','white'),('Black','black')], value='white', description='You:')
        self.start_btn = W.Button(description='Start / Reset', button_style='primary')
        self.undo_btn = W.Button(description='Undo')
        self.flip_btn = W.Button(description='Flip')
        self.status = W.HTML("<b>Click Start / Reset to begin.</b>")

        # Moves log
        self.log = W.HTML("")
        self.log_box = W.VBox([W.HTML("<b>Moves</b>"),
                               W.Box([self.log], layout=W.Layout(max_height='220px', overflow='auto'))])

        # AI analysis box
        self.analysis = W.HTML(render_analysis_panel(chess.Board()))
        self.analysis_box = W.Box(
    [self.analysis],
    layout=W.Layout(
        width='360px',
        max_height='420px',      # cap the height
        overflow='auto',
        align_self='flex-start'  # don't stretch
    )
)
        
        self.log_box.layout = W.Layout(width='220px', align_self='flex-start')
        
        self.side_panel = W.VBox([W.HTML("<b>AI Analysis</b>"), self.analysis])

        # Promotion chooser (hidden until needed)
        self.promo_box = W.HBox([])
        self._promo_pending = None  # (from_sq, to_sq)

        # Board grid
        self.grid_box = W.GridBox(children=[], layout=W.Layout(grid_template_columns='repeat(8, 46px)', grid_gap='0px'))
        self.sq_buttons: Dict[int, W.Button] = {}
        self.selected_sq: Optional[int] = None
        self.legal_from_selected: List[chess.Move] = []

        # State
        self.state: Optional[GameState] = None

        # Wire controls
        self.start_btn.on_click(self.on_start)
        self.undo_btn.on_click(self.on_undo)
        self.flip_btn.on_click(self.on_flip)
        self.mode.observe(self.on_mode_change, 'value')

        # Build panel
        top = W.HBox([self.mode, self.color_sel, self.start_btn, self.undo_btn, self.flip_btn])
        board_row = W.HBox(
    [self.grid_box, self.log_box, self.analysis_box],
    layout=W.Layout(
        align_items='flex-start',        # <- key: no stretching
        justify_content='flex-start',
        column_gap='12px'
    )
)
        self.ui = W.VBox([top, self.status, self.promo_box, board_row])

        self._build_empty_grid()
        display(self.ui)

    # ---------- helpers ----------
    def _square_to_rc(self, sq: int) -> Tuple[int,int]:
        file = chess.square_file(sq); rank = chess.square_rank(sq)
        if self.state and self.state.orientation_white: row, col = 7 - rank, file
        else:                                           row, col = rank, 7 - file
        return row, col

    def _rc_to_square(self, row: int, col: int) -> int:
        if self.state and self.state.orientation_white: rank, file = 7 - row, col
        else:                                           rank, file = row, 7 - col
        return chess.square(file, rank)

    def _build_empty_grid(self):
        self.grid_box.children = ()
        self.sq_buttons.clear()
        children = []
        for row in range(8):
            for col in range(8):
                sq = self._rc_to_square(row, col)
                light = (row + col) % 2 == 0
                b = W.Button(
    description=' ',
    layout=W.Layout(width='46px', height='46px', min_width='46px', min_height='46px', padding='0'),
    tooltip=chess.square_name(sq)
)

                b.style.button_color = LIGHT if light else DARK
                b.on_click(self._make_square_click(sq))
                self.sq_buttons[sq] = b
                children.append(b)
        self.grid_box.children = tuple(children)

    def _render_board(self):
        board = self.state.board
        # Clear selection if no longer legal
        if self.selected_sq is not None and (self.selected_sq not in [m.from_square for m in board.legal_moves]):
            self.selected_sq = None
            self.legal_from_selected = []
        # Set colors + glyphs
        for sq, btn in self.sq_buttons.items():
            row, col = self._square_to_rc(sq)
            light = (row + col) % 2 == 0
            btn.style.button_color = LIGHT if light else DARK
            piece = board.piece_at(sq)
            btn.description = GLYPH.get(piece.symbol(), ' ') if piece else ' '
        # Highlights
        if self.selected_sq is not None:
            self.sq_buttons[self.selected_sq].style.button_color = SEL
            for m in self.legal_from_selected:
                tgt_btn = self.sq_buttons[m.to_square]
                tgt_btn.style.button_color = CAPT if self.state.board.is_capture(m) else TARGET

        # Status & Analysis
        self.status.value = f"<b>{self.state.status_text()}</b>"
        self.analysis.value = render_analysis_panel(self.state.board)

    def _append_log(self, text: str):
        self.log.value += f"{text}<br>"
        if self.log.value.count("<br>") > 200:
            self.log.value = "<i>(trimmed)</i><br>" + "<br>".join(self.log.value.split("<br>")[-200:])

    # ---------- interactions ----------
    def _make_square_click(self, sq: int):
        def handler(_):
            if self.state is None or self.state.board.is_game_over(): return
            if self._promo_pending is not None: return  # wait for promo choice

            board = self.state.board
            # enforce side-to-move if Vs AI
            if self.state.ai_color is not None:
                human_turn = (board.turn != self.state.ai_color)
                if not human_turn: return

            piece = board.piece_at(sq)
            if self.selected_sq is None:
                if piece is None or piece.color != board.turn: return
                self.selected_sq = sq
                self.legal_from_selected = [m for m in board.legal_moves if m.from_square == sq]
                self._render_board(); return
            else:
                if sq == self.selected_sq:
                    self.selected_sq = None; self.legal_from_selected = []; self._render_board(); return
                if piece is not None and piece.color == board.turn:
                    self.selected_sq = sq
                    self.legal_from_selected = [m for m in board.legal_moves if m.from_square == sq]
                    self._render_board(); return

                legal_targets = [m for m in self.legal_from_selected if m.to_square == sq]
                if not legal_targets: return

                promos = [m for m in legal_targets if m.promotion]
                if promos:
                    self._prompt_promotion(self.selected_sq, sq, promos); return

                mv = legal_targets[0]
                self._play_human_move(mv)
        return handler

    def _prompt_promotion(self, from_sq: int, to_sq: int, promo_moves: List[chess.Move]):
        self._promo_pending = (from_sq, to_sq)
        opts = [('Queen','q'),('Rook','r'),('Bishop','b'),('Knight','n')]
        picker = W.ToggleButtons(options=opts, value='q', description='Promote:')
        ok_btn = W.Button(description='OK', button_style='success')
        cancel_btn = W.Button(description='Cancel')
        msg = W.HTML("")

        def on_ok(_):
            letter = picker.value; promo_map = {'q':chess.QUEEN,'r':chess.ROOK,'b':chess.BISHOP,'n':chess.KNIGHT}
            promo_piece = promo_map[letter]
            for m in promo_moves:
                if m.from_square == from_sq and m.to_square == to_sq and m.promotion == promo_piece:
                    self._clear_promo_ui(); self._play_human_move(m); return
            msg.value = "<span style='color:#b91c1c'>Not a legal promotion.</span>"

        def on_cancel(_):
            self._clear_promo_ui()

        ok_btn.on_click(on_ok); cancel_btn.on_click(on_cancel)
        self.promo_box.children = [picker, ok_btn, cancel_btn, msg]

    def _clear_promo_ui(self):
        self.promo_box.children = []; self._promo_pending = None

    # ---------- actions ----------
    def on_start(self, _):
        human_color = self.color_sel.value
        ai = (None if self.mode.value == '2p'
              else (chess.BLACK if human_color == 'white' else chess.WHITE))
        self.state = GameState(board=chess.Board(), ai_color=ai,
                               orientation_white=True if human_color=='white' else False)
        self.selected_sq = None; self.legal_from_selected = []
        self._build_empty_grid(); self.log.value = ""; self._clear_promo_ui()

        # If AI to move first
        if self.state.ai_color == chess.WHITE and self.state.board.turn == chess.WHITE:
            mv = best_move(self.state.board, self.state.ai_color)
            if mv:
                san = self.state.board.san(mv)
                self.state.board.push(mv); self._append_log(f"AI: {san}")
        self._render_board()

    def on_undo(self, _):
        if not self.state: return
        b = self.state.board; undone = 0
        if len(b.move_stack): b.pop(); undone += 1
        if self.state.ai_color is not None and len(b.move_stack): b.pop(); undone += 1
        self.selected_sq = None; self.legal_from_selected = []
        self._append_log(f"<i>Undid {undone} half-move(s).</i>")
        self._render_board()

    def on_flip(self, _):
        if not self.state: return
        self.state.orientation_white = not self.state.orientation_white
        self._build_empty_grid(); self._render_board()

    def on_mode_change(self, change):
        self.color_sel.disabled = (change['new'] != 'ai')

    def _play_human_move(self, mv: chess.Move):
        b = self.state.board
        san = b.san(mv); b.push(mv); self._append_log(f"You: {san}")
        self.selected_sq = None; self.legal_from_selected = []
        self._render_board()
        if not b.is_game_over() and self.state.ai_color is not None and b.turn == self.state.ai_color:
            reply = best_move(b, self.state.ai_color)
            if reply:
                san2 = b.san(reply); b.push(reply); self._append_log(f"AI: {san2}")
                self._render_board()

# Launch the app
ChessApp()


## How to Use

1. **Start a Game**: Click "Start / Reset" button
2. **Select Mode**: Choose "Vs AI" or "Two Players"
3. **Choose Color**: If playing AI, select White or Black
4. **Make Moves**: Click a piece to select it, then click a target square
5. **Pawn Promotion**: Choose a piece when a pawn reaches the last rank
6. **View Analysis**: Check the AI Analysis panel for move suggestions and scores
7. **Undo**: Click "Undo" to take back moves
8. **Flip Board**: Click "Flip" to change perspective